# Train TRPO on continuous actions

TRPO updates a Gaussian actor using fixed generalized advantage estimates (GAE):

$$\max_\theta\frac1N\sum_t\rho_t\hat A_t\quad\mathrm{subject\ to}\quad\frac1N\sum_t D_{KL}(\pi_{old}\|\pi_\theta)\le\delta.$$

Here $\rho_t=\pi_\theta(a_t\mid s_t)/\pi_{old}(a_t\mid s_t)$ is the probability ratio, $\hat A_t$ the advantage, and $N$ the rollout size. The mean KL budget is $\delta$. We train on `Pendulum-v1` with lower gravity (`g=1.0`) for a shorter CPU demonstration. A tanh transform and affine rescaling keep actions within the torque bounds. TRPO computes analytic KL between the underlying diagonal Gaussians; the shared action transform preserves KL.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import TRPO, TRPOConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "Pendulum-v1"
ENV_KWARGS = {"g": 1.0}

In [ ]:
env = gym.make(ENV_ID, **ENV_KWARGS)
try:
    config = TRPOConfig(
        max_kl=0.01,
        cg_steps=10,
        value_epochs=10,
        value_learning_rate=1e-3,
        n_steps=1024,
        gae_lambda=0.95,
        normalize_advantage=True,
        seed=7,
    )
    agent = TRPO(env, config=config, device="cpu")
    agent.learn(total_timesteps=51_200)
finally:
    env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(10, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"TRPO training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(
    ENV_ID, render_mode="human", **ENV_KWARGS
)
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True, seed=1_000
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")